In [38]:
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path

In [39]:
SCRIPTS = Path.cwd()
DATA = SCRIPTS.parent / "data"

In [ ]:
acc_file = "joel_appal_acc_260609_1014.csv"
ecg_file = "joel_appal_ecg_260609_1014.csv" 

acc_df = pd.read_csv(DATA / acc_file)
ecg_df = pd.read_csv(DATA / ecg_file)

In [41]:
ecg_df.shape, acc_df.shape

((10074, 7), (15804, 9))

In [42]:
ecg_df.head(), acc_df.head()

(   packet_id  device_time_ms  host_time_ms   ecg  sample_idx  sample_time_ms  \
 0          0    5.996161e+11  1.780991e+12  -429           0    5.996161e+11   
 1          0    5.996161e+11  1.780991e+12 -1118           1    5.996161e+11   
 2          0    5.996161e+11  1.780991e+12 -1386           2    5.996161e+11   
 3          0    5.996161e+11  1.780991e+12 -1026           3    5.996161e+11   
 4          0    5.996161e+11  1.780991e+12  -837           4    5.996161e+11   
 
      time_ms  
 0   0.000000  
 1   7.692383  
 2  15.384644  
 3  23.076904  
 4  30.769287  ,
    packet_id  device_time_ms  host_time_ms   x    y    z  sample_idx  \
 0          0    5.996161e+11  1.780991e+12 -77 -343 -981           0   
 1          0    5.996161e+11  1.780991e+12 -77 -341 -977           1   
 2          0    5.996161e+11  1.780991e+12 -80 -337 -985           2   
 3          0    5.996161e+11  1.780991e+12 -79 -342 -983           3   
 4          0    5.996161e+11  1.780991e+12 -77 -3

In [43]:
ecg_df = ecg_df.sort_values("time_ms")
acc_df = acc_df.sort_values("time_ms")

recording = pd.merge_asof(
    ecg_df,
    acc_df[["time_ms", "x", "y", "z"]],
    on="time_ms",
    direction="nearest"
)
recording = recording[["time_ms", "ecg", "x", "y", "z"]]
recording["acc_mag"] = np.sqrt(
    recording["x"]**2 +
    recording["y"]**2 +
    recording["z"]**2)

In [44]:
recording

,time_ms,ecg,x,y,z,acc_mag
0,0.000000,-429,-77,-343,-981,1042.083970
1,7.692383,-1118,-80,-337,-985,1044.123556
2,15.384644,-1386,-79,-342,-983,1043.788293
3,23.076904,-1026,-81,-343,-982,1043.328328
4,30.769287,-837,-80,-340,-982,1042.268679
...,...,...,...,...,...,...
10069,77464.076782,-159,-28,-551,-849,1012.514691
10070,77471.769165,-96,-28,-551,-849,1012.514691
10071,77479.461426,-117,-28,-551,-849,1012.514691
10072,77487.153809,-152,-28,-551,-849,1012.514691


In [45]:
import neurokit2 as nk

fs = 130

_, info = nk.ecg_peaks(
    recording["ecg"],
    sampling_rate=fs
)

rpeaks = info["ECG_R_Peaks"]

recording["r_peak"] = 0
recording.loc[rpeaks, "r_peak"] = 1

In [46]:
recording

,time_ms,ecg,x,y,z,acc_mag,r_peak
0,0.000000,-429,-77,-343,-981,1042.083970,0
1,7.692383,-1118,-80,-337,-985,1044.123556,0
2,15.384644,-1386,-79,-342,-983,1043.788293,0
3,23.076904,-1026,-81,-343,-982,1043.328328,0
4,30.769287,-837,-80,-340,-982,1042.268679,0
...,...,...,...,...,...,...,...
10069,77464.076782,-159,-28,-551,-849,1012.514691,0
10070,77471.769165,-96,-28,-551,-849,1012.514691,0
10071,77479.461426,-117,-28,-551,-849,1012.514691,0
10072,77487.153809,-152,-28,-551,-849,1012.514691,0


In [58]:
segment = recording.query(
    "50000 <= time_ms <= 60000"
)

peaks = segment[segment["r_peak"] == 1]

fig = px.line(
    segment,
    x="time_ms",
    y="ecg"
)

fig.add_scatter(
    x=peaks["time_ms"],
    y=peaks["ecg"],
    mode="markers",
    name="R peaks"
)

fig.show()

In [48]:
import numpy as np
import pandas as pd

# R-peak times
rpeak_times = recording.loc[recording["r_peak"] == 1, "time_ms"].to_numpy()

# -----------------------------
# 1) Beat-to-beat HR / HRV
# -----------------------------
rr_ms = np.diff(rpeak_times)

hrv_df = pd.DataFrame({
    "time_ms": rpeak_times[1:],
    "rr_ms": rr_ms,
    "bpm_instant": 60000 / rr_ms
})

# -----------------------------
# 2) 1-second sport-watch-like BPM
# -----------------------------
bpm_df = (
    hrv_df
    .set_index(pd.to_timedelta(hrv_df["time_ms"], unit="ms"))
    .resample("1s")
    .mean(numeric_only=True)
    .interpolate()
    .reset_index(drop=True)
)

bpm_df["time_ms"] = np.arange(len(bpm_df)) * 1000
bpm_df["bpm"] = bpm_df["bpm_instant"].rolling(
    window=5,
    center=True,
    min_periods=1
).mean()

bpm_df = bpm_df[["time_ms", "bpm"]]

In [49]:
px.line(
    hrv_df,
    x="time_ms",
    y="rr_ms",
    title="RR Intervals Over Time"
)

In [50]:
px.line(
    bpm_df,
    x="time_ms",
    y="bpm",
    title="BPM Over Time"
)

In [51]:
px.line(
    hrv_df,
    x="time_ms",
    y="rr_ms",
    title="RR Intervals Over Time"
)

In [52]:
rr = hrv_df["rr_ms"]

px.scatter(
    x=rr[:-1],
    y=rr[1:],
    title="Poincaré of RR Intervals",
    labels={"x": "RR(n) [ms]", "y": "RR(n+1) [ms]"}
)

In [53]:
rr_diff = np.diff(hrv_df["rr_ms"])

hrv_df["rmssd_30"] = (
    pd.Series(rr_diff**2)
    .rolling(30)
    .mean()
    .pow(0.5)
)

px.line(
    hrv_df.iloc[1:],
    x="time_ms",
    y="rmssd_30",
    title="HRV Over Time"
)